# StyleGAN2 Latent Space Lab

This notebook demonstrates how to load a pretrained StyleGAN2 generator, sample latent vectors (z), generate images, and perform simple vector arithmetic in latent space to observe effects on generated images. The model file is expected at `download/ffhq.pkl`.

In [ ]:
# Imports and helpers
import pickle as pkl
import sys
import torch
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
import os
import uuid

plt.rcParams['figure.figsize'] = [6,6]

# helper to convert synthesis output to PIL image
def synth_to_pil(img_tensor):
    # img_tensor: [B, C, H, W] float32 in range ~[-1,1]
    img = (img_tensor * 127.5 + 128).clamp(0, 255).to(torch.uint8).cpu().numpy()
    # take first image
    arr = img[0].transpose(1,2,0)
    return Image.fromarray(arr)

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# Load StyleGAN2 generator and pretrained ffhq model

In [ ]:
if torch.cuda.is_available():
    device = torch.device("cuda")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")


with open('download/ffhq.pkl', 'rb') as f:
    G = pkl.load(f)['G_ema'].to(device)
G.eval()
print('Loaded generator successfully. z_dim =', G.z_dim, 'img_resolution =', G.img_resolution)


### 2 helper functions: 1) sample a z, and 2) generate image from z

In [ ]:
# Sample a random latent z and generate an image
def sample_z():
    return torch.randn([1, G.z_dim], device=device)

def z_to_image(z, save_path=None):
    with torch.no_grad():
        c = None
        w = G.mapping(z, c)
        # disable per-layer noise for reproducibility here
        img_tensor = G.synthesis(w)
        pil = synth_to_pil(img_tensor)
        if save_path is not None:
            ensure_dir(os.path.dirname(save_path))
            pil.save(save_path)
        return pil, w, img_tensor

# Example: generate one image, display at full sizes 
z = sample_z()
pil, w, img_tensor = z_to_image(z)
display(pil)
